In [ ]:
# imports
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
DIR = Path("dataset")
files = sorted(DIR.rglob("*.parquet"))
print(len(files), "files")

2228 files


In [ ]:
def load_instance(path: Path) -> pd.DataFrame:
    """Read each instance of Parquet file to add metadata"""
    df = pd.read_parquet(path)

    if isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()

    # Adding metadata
    file_name = path.stem.upper()
    df["instance_id"] = path.stem
    df["event_class"] = int(path.parent.name)

    # WELL = real well, SIMULATED = simulation, DRAWN = handdrawn
    df["source"] = ("simulated" if file_name.startswith("SIMULATED")
                    else "drawn" if file_name.startswith("DRAWN")
                    else "real")

    df["well_id"] = path.stem.split("_")[0] if file_name.startswith("WELL") else None

    return df


def reduce_memory(df: pd.DataFrame, save_path: str) -> pd.DataFrame:

    print(f"Total memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
    
    # float32
    for c in df.select_dtypes("float64").columns:
        df[c] = df[c].where(df[c].abs() <= np.finfo("float32").max).astype("float32")

    # metadata text to category
    for c in ["instance_id", "well_id", "source"]:
        df[c] = df[c].astype("category")
    df["event_class"] = df["event_class"].astype("int8")

    print(f"Reduced memory usage: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

    df.to_parquet(save_path)
    return df


CACHE = Path("final.parquet")

if CACHE.exists():
    df = pd.read_parquet(CACHE)
    print("df loaded from cache.")
else:
    frames = []
    for i, f in enumerate(files):
        frames.append(load_instance(f))
        if i % 500 == 0:                     
            print(f"{i}/{len(files)}")

    df = pd.concat(frames, ignore_index=True)
    df = reduce_memory(df, save_path="final.parquet")
    del frames
    print("Loading completed.")


print(f"Shape: {df.shape}")
print(df.dtypes)
df.head()

ValueError: invalid literal for int() with base 10: 'WELL-00001_20170201010207'

In [ ]:
# --- 1. Quante istanze ci sono per ogni classe di evento e per ogni sorgente ---
# groupby su due chiavi (classe, sorgente); nunique() conta le istanze distinte,
# non le righe. unstack() sposta le sorgenti sulle colonne -> tabella classi x sorgenti.
# fill_value=0 mette 0 dove una combinazione non esiste (es. classe 0 simulata).
# Serve a vedere lo sbilanciamento e quante istanze reali abbiamo per ogni guasto.
print(df.groupby(["event_class", "source"], observed=True)["instance_id"]
        .nunique().unstack(fill_value=0))

# --- 2. Valori mancanti: quali colonne hanno più NaN ---
# isna() segna i NaN, mean() dà la frazione per colonna (0 = nessun NaN, 1 = tutta vuota).
# Ordino dal peggiore e mostro le 15 colonne più vuote: sono i sensori problematici
# da analizzare meglio nella heatmap dei missing.
print(df.isna().mean().sort_values(ascending=False).head(15))

# --- 3. Lunghezza di ogni istanza (numero di righe = numero di timestamp) ---
# size() conta le righe di ogni istanza; describe() riassume la distribuzione
# (media, minimo, massimo, quartili). Serve a vedere se le istanze sono
# comparabili e se ce ne sono di troppo corte per essere divise in finestre.
lens = df.groupby("instance_id", observed=True).size()
print(lens.describe())
